# 04. Classification Training & MLflow Tracking — EMIPredict AI

This notebook trains 4 classification algorithms (Logistic Regression, Random Forest, Gradient Boosting, and XGBoost) to predict `emi_eligibility` (3 classes: `Eligible`, `High_Risk`, `Not_Eligible`) using the 48 scaled and engineered features. All experiments are logged to MLflow (hosted on DagsHub with fallback to local `mlruns/`).

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn
import mlflow.xgboost
from mlflow.models.signature import infer_signature

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# MLflow Setup (DagsHub Remote with fallback to local mlruns)
try:
    from google.colab import userdata
    dagshub_uri = userdata.get('DAGSHUB_MLFLOW_URI') or userdata.get('MLFLOW_TRACKING_URI')
    dagshub_user = userdata.get('DAGSHUB_USERNAME') or userdata.get('MLFLOW_TRACKING_USERNAME')
    dagshub_token = userdata.get('DAGSHUB_TOKEN') or userdata.get('MLFLOW_TRACKING_PASSWORD')
    if dagshub_uri:
        os.environ['MLFLOW_TRACKING_URI'] = dagshub_uri
    if dagshub_user:
        os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_user
    if dagshub_token:
        os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token
except Exception:
    pass

tracking_uri = os.environ.get('MLFLOW_TRACKING_URI')
if not tracking_uri:
    mlruns_dir = '../mlruns' if os.path.exists('../mlruns') or os.path.exists('../notebooks') else 'mlruns'
    os.makedirs(mlruns_dir, exist_ok=True)
    tracking_uri = f"file:{os.path.abspath(mlruns_dir)}"

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("emi_classification")
print(f"MLflow tracking configured at: {mlflow.get_tracking_uri()}")

MLflow tracking configured at: https://dagshub.com/dummy2357dummy/EMI_Prediction.mlflow


In [2]:
# Load Engineered Dataset (48 Features)
data_path = '../data/processed/engineered_dataset.csv' if os.path.exists('../data') else 'data/processed/engineered_dataset.csv'
df = pd.read_csv(data_path)

feature_cols = [
    'age', 'monthly_salary', 'years_of_employment', 'monthly_rent', 'family_size',
    'dependents', 'school_fees', 'college_fees', 'travel_expenses', 'groceries_utilities',
    'other_monthly_expenses', 'existing_loans', 'current_emi_amount', 'credit_score',
    'bank_balance', 'emergency_fund', 'requested_amount', 'requested_tenure',
    'debt_to_income_ratio', 'expense_to_income_ratio', 'affordability_ratio', 'risk_score',
    'salary_credit_interaction', 'surplus_to_requested_ratio',
    'gender_Female', 'gender_Male',
    'marital_status_Married', 'marital_status_Single',
    'education_Graduate', 'education_High School', 'education_Post Graduate', 'education_Professional',
    'employment_type_Government', 'employment_type_Private', 'employment_type_Self-employed',
    'company_type_Large Indian', 'company_type_MNC', 'company_type_Mid-size', 'company_type_Small', 'company_type_Startup',
    'house_type_Family', 'house_type_Own', 'house_type_Rented',
    'emi_scenario_E-commerce Shopping', 'emi_scenario_Education', 'emi_scenario_Home Appliances', 'emi_scenario_Personal Loan', 'emi_scenario_Vehicle'
]

# Encode Target
target_map = {'Eligible': 0, 'High_Risk': 1, 'Not_Eligible': 2}
reverse_map = {0: 'Eligible', 1: 'High_Risk', 2: 'Not_Eligible'}
df['target_code'] = df['emi_eligibility'].map(target_map)

train_df = df[df['dataset_split'] == 'train']
val_df = df[df['dataset_split'] == 'val']

X_train, y_train = train_df[feature_cols].values, train_df['target_code'].values
X_val, y_val = val_df[feature_cols].values, val_df['target_code'].values

print(f"Train shape: {X_train.shape} (48 features), Validation shape: {X_val.shape}")


Train shape: (283360, 48) (48 features), Validation shape: (60720, 48)


In [3]:
# Model Candidates Definition
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42),
    "Gradient Boosting Classifier": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42),
    "XGBoost Classifier": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, eval_metric='mlogloss', random_state=42)
}

# Training & Logging Loop
for name, model in models.items():
    with mlflow.start_run(run_name=name):
        print(f"Training {name}...")
        model.fit(X_train, y_train)
        
        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val) if hasattr(model, "predict_proba") else None
        
        acc = accuracy_score(y_val, y_pred)
        prec = precision_score(y_val, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_val, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
        roc_auc = roc_auc_score(y_val, y_proba, multi_class='ovr', average='macro') if y_proba is not None else 0.0
        
        mlflow.log_params(model.get_params() if hasattr(model, 'get_params') else {})
        mlflow.log_metrics({
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc": roc_auc
        })
        
        # Confusion Matrix Artifact
        cm = confusion_matrix(y_val, y_pred)
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Eligible', 'High_Risk', 'Not_Eligible'], yticklabels=['Eligible', 'High_Risk', 'Not_Eligible'])
        plt.title(f'Confusion Matrix - {name}')
        cm_path = f"cm_{name.lower().replace(' ', '_')}.png"
        plt.tight_layout()
        plt.savefig(cm_path)
        plt.close()
        mlflow.log_artifact(cm_path)
        if os.path.exists(cm_path):
            os.remove(cm_path)
        
        # Log Model with Signature
        signature = infer_signature(X_val, y_pred)
        if "XGB" in name:
            mlflow.xgboost.log_model(model, artifact_path="model", signature=signature)
        else:
            mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
            
        print(f"Successfully logged {name} to MLflow (F1: {f1:.4f}, Accuracy: {acc:.4f})")

Training Logistic Regression...


Successfully logged Logistic Regression to MLflow (F1: 0.5867, Accuracy: 0.9012)


Training Random Forest Classifier...


Successfully logged Random Forest Classifier to MLflow (F1: 0.6607, Accuracy: 0.9495)


Training Gradient Boosting Classifier...


Successfully logged Gradient Boosting Classifier to MLflow (F1: 0.8284, Accuracy: 0.9635)


Training XGBoost Classifier...


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:160: UserWarning: [15:37:51] WARNING: /workspace/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
  warnings.warn(smsg, UserWarning)


Successfully logged XGBoost Classifier to MLflow (F1: 0.8975, Accuracy: 0.9764)
